# ChemSPAR quick start

This notebook loads a crystal, prunes it with two orderings under the protection gate, compares how many protected
contacts survive with and without the gate, inspects the audit log, and exports a graph view to PyTorch Geometric.

It uses a small **synthetic** test crystal shipped with the repository (`tests/data/synthetic/`) so that it runs in
seconds. Replace `CIF` with any CIF of your own, e.g. a QMOF structure.

In [1]:
from pathlib import Path

import chemspar
from chemspar import build_graph, orderings, prune

CIF = Path("../tests/data/synthetic/syn-cu-formate-water.cif")
chemspar.__version__

'1.0.0'

## 1. Build the periodic contact graph
Every atom pair within 4.5 Å is one edge per lattice image (a *contact instance*). Each instance gets a bond class
from the taxonomy; three classes are protected.

In [2]:
g = build_graph(CIF, cutoff=4.5)
print(len(g), "contact instances,", len(g.protected), "protected")
g.bond_class_counts()

145 contact instances, 17 protected


{'metal_contact': 12,
 'metal_ligand': 6,
 'other_contact': 116,
 'covalent_radius': 11}

## 2. Prune with two orderings, gated and ungated
`tau` is the removal budget as a fraction of all instances.

In [3]:
import pandas as pd

rows = []
for ordering in (orderings.distance, orderings.gravity):
    for gate in (True, False):
        view, log = prune(g, ordering=ordering, tau=0.5, gate=gate)
        rows.append(dict(ordering=ordering.name, gate=gate, removed=len(view.removed), target=view.target,
                         rejected=log.rejected, protected_retention=round(view.protected_retention(), 3),
                         connected=view.is_connected()))
pd.DataFrame(rows)

,ordering,gate,removed,target,rejected,protected_retention,connected
0,distance,True,72,72,0,1.000,True
1,distance,False,72,72,0,1.000,True
2,gravity,True,72,72,3,1.000,True
3,gravity,False,72,72,3,0.941,True


Under the gate every protected contact is retained (`protected_retention = 1.0`). The ungated partner can
remove protected contacts; it keeps only the degree and connectivity checks.

## 3. Inspect the audit log
Every attempt is one row, written before the decision, with the outcome of each gate check and the reason.

In [4]:
view, log = prune(g, ordering="distance", tau=0.5, gate=True)
df = log.to_dataframe()
df[["rank", "element_i", "element_j", "distance", "bond_class", "protection_route",
    "check_protected", "check_degree", "check_connectivity", "decision", "reason"]].head(10)

,rank,element_i,element_j,distance,bond_class,protection_route,check_protected,check_degree,check_connectivity,decision,reason
0,0,H,H,4.400420,other_contact,none,pass,pass,pass,accepted,accepted
1,1,H,H,4.400420,other_contact,none,pass,pass,pass,accepted,accepted
2,2,H,H,4.400420,other_contact,none,pass,pass,pass,accepted,accepted
3,3,H,H,4.400420,other_contact,none,pass,pass,pass,accepted,accepted
4,4,H,C,4.337246,other_contact,none,pass,pass,pass,accepted,accepted
5,5,H,C,4.337246,other_contact,none,pass,pass,pass,accepted,accepted
6,6,H,C,4.337246,other_contact,none,pass,pass,pass,accepted,accepted
7,7,H,C,4.337246,other_contact,none,pass,pass,pass,accepted,accepted
8,8,H,O,4.306704,other_contact,none,pass,pass,pass,accepted,accepted
9,9,H,O,4.306704,other_contact,none,pass,pass,pass,accepted,accepted


In [5]:
log.summary()

{'structure_id': 'syn-cu-formate-water',
 'policy': 'distance',
 'gate_mode': 'corrected',
 'tau': 0.5,
 'target': 72,
 'attempts': 72,
 'removed': 72,
 'rejected': 0,
 'rejected_by_check': {},
 'rejected_by_reason': {},
 'removed_by_bond_class': {'other_contact': 72}}

In [6]:
log.to_parquet("audit_distance_chem_0.5.parquet")   # or log.to_csv(...)

## 4. Bring your own ordering
Any ordering works: a callable returning instance keys, a list of keys, a rank array, or scores.

In [7]:
# shortest contacts first, written as scores (ascending = removed first)
scores = [g.distance(k) for k in g.instances]
view_s, log_s = prune(g, ordering=orderings.from_scores(scores), tau=0.2, gate=True)
log_s.summary()["rejected_by_check"]

{'protected': 17}

## 5. Export
The PyG `Data` object is built directly from the retained instances: each instance becomes two directed edges with
its distance and lattice image. Requires `pip install "chemspar[pyg]"`.

In [8]:
data = view.to_pyg()
data

Data(edge_index=[2, 146], z=[16], roles=[16, 7], edge_dist=[146], edge_image=[146, 3], structure_id='syn-cu-formate-water', num_nodes=16)

In [9]:
import json

d = json.loads(view.to_json())
{k: d[k] for k in ("structure_id", "policy", "tau", "target", "n_instances")}, len(d["retained"])

({'structure_id': 'syn-cu-formate-water',
  'policy': 'distance',
  'tau': 0.5,
  'target': 72,
  'n_instances': 145},
 73)